In [1]:
!pip install -q imagehash

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 296.7/296.7 kB 26.2 MB/s eta 0:00:00


In [2]:
import os
import zipfile
import random
import numpy as np
import tensorflow as tf
import cv2
import imagehash
from tqdm import tqdm
from PIL import Image

from collections import defaultdict, Counter
from sklearn.model_selection import train_test_split

import matplotlib.pyplot as plt
from tensorflow.keras import layers

In [3]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [4]:
zip_path = "/content/drive/MyDrive/Parkinson's Dataset Final.zip"
extract_path = "/content/dataset"

os.makedirs(extract_path, exist_ok=True)

with zipfile.ZipFile(zip_path, "r") as z:
    z.extractall(extract_path)

BASE = "/content/dataset/Parkinson's Dataset Final"

print(os.listdir(BASE))

['New hand pd Dataset', 'Hand Pd']


In [5]:
HANDPD_ROOT = os.path.join(BASE, "Hand Pd")

SPIRAL_ROOT = os.path.join(HANDPD_ROOT, "Spiral_HandPD")
MEANDER_ROOT = os.path.join(HANDPD_ROOT, "Meander_HandPD")

image_paths=[]
labels=[]

def collect(root):

    for cls in os.listdir(root):

        cls_path=os.path.join(root,cls)

        if not os.path.isdir(cls_path):
            continue

        if cls.lower()=="healthy":
            lab=0
        elif cls.lower()=="parkinson":
            lab=1
        else:
            continue

        for r,d,files in os.walk(cls_path):

            for f in files:

                if f.lower().endswith((".png",".jpg",".jpeg",".bmp")):

                    image_paths.append(os.path.join(r,f))
                    labels.append(lab)

collect(SPIRAL_ROOT)
collect(MEANDER_ROOT)

print("Total images:",len(image_paths))

Total images: 808


In [6]:
def get_phash(path):

    try:

        img=Image.open(path).convert("L").resize((224,224))

        return imagehash.phash(img)

    except:
        return None


hash_map=defaultdict(list)

for p in tqdm(image_paths):

    h=get_phash(p)

    if h is not None:

        hash_map[str(h)].append(p)


clean_paths=[]
clean_labels=[]

path_to_label=dict(zip(image_paths,labels))

for h in hash_map:

    p=hash_map[h][0]

    clean_paths.append(p)
    clean_labels.append(path_to_label[p])

print("Clean dataset:",len(clean_paths))

100%|██████████| 808/808 [00:08<00:00, 94.00it/s] 

Clean dataset: 735


In [7]:
paths=np.array(clean_paths)
labels=np.array(clean_labels)

train_paths,test_paths,y_train,y_test=train_test_split(
paths,
labels,
test_size=0.2,
stratify=labels,
random_state=42
)

val_paths,test_paths,y_val,y_test=train_test_split(
test_paths,
y_test,
test_size=0.5,
stratify=y_test,
random_state=42
)

print("Train:",Counter(y_train))
print("Val:",Counter(y_val))
print("Test:",Counter(y_test))

Train: Counter({np.int64(1): 473, np.int64(0): 115})
Val: Counter({np.int64(1): 59, np.int64(0): 14})
Test: Counter({np.int64(1): 59, np.int64(0): 15})


In [10]:
from sklearn.utils.class_weight import compute_class_weight
import numpy as np

class_weights = compute_class_weight(
    class_weight="balanced",
    classes=np.unique(y_train),
    y=y_train
)

class_weights = dict(enumerate(class_weights))

print(class_weights)

{0: np.float64(2.5565217391304347), 1: np.float64(0.6215644820295984)}


In [11]:
IMG_SIZE=224
BATCH_SIZE=16

def preprocess(path):

    path=path.numpy().decode()

    img=cv2.imread(path,cv2.IMREAD_GRAYSCALE)

    img=cv2.resize(img,(IMG_SIZE,IMG_SIZE))

    clahe=cv2.createCLAHE(clipLimit=2.0,tileGridSize=(8,8))

    img=clahe.apply(img)

    img=img.astype(np.float32)/255.0

    img=np.expand_dims(img,-1)

    return img


def tf_preprocess(path,label):

    img=tf.py_function(preprocess,[path],tf.float32)

    img.set_shape((IMG_SIZE,IMG_SIZE,1))

    return img,label

In [12]:
data_augmentation = tf.keras.Sequential([

layers.RandomFlip("horizontal"),

layers.RandomRotation(0.15),

layers.RandomZoom(0.1),

layers.RandomContrast(0.1)

])

In [13]:
def make_train_ds(paths,labels):

    ds=tf.data.Dataset.from_tensor_slices((paths,labels))

    ds=ds.shuffle(2000)

    ds=ds.map(tf_preprocess,num_parallel_calls=tf.data.AUTOTUNE)

    ds=ds.batch(BATCH_SIZE)

    ds=ds.map(lambda x,y:(data_augmentation(x,training=True),y))

    ds=ds.prefetch(tf.data.AUTOTUNE)

    return ds


def make_eval_ds(paths,labels):

    ds=tf.data.Dataset.from_tensor_slices((paths,labels))

    ds=ds.map(tf_preprocess,num_parallel_calls=tf.data.AUTOTUNE)

    ds=ds.batch(BATCH_SIZE)

    ds=ds.prefetch(tf.data.AUTOTUNE)

    return ds


train_ds=make_train_ds(train_paths,y_train)
val_ds=make_eval_ds(val_paths,y_val)
test_ds=make_eval_ds(test_paths,y_test)

In [20]:
PATCH_SIZE = 16
EMBED_DIM = 256
NUM_PATCHES = (IMG_SIZE // PATCH_SIZE) ** 2


class PatchEmbedding(layers.Layer):

    def __init__(self, patch_size, embed_dim):
        super().__init__()

        self.proj = layers.Conv2D(embed_dim, patch_size, patch_size)

    def call(self, x):

        x = self.proj(x)

        x = tf.reshape(x, [tf.shape(x)[0], -1, EMBED_DIM])

        return x


class AddPositionEmbedding(layers.Layer):

    def __init__(self, num_patches, embed_dim):
        super().__init__()

        self.num_patches = num_patches
        self.embed_dim = embed_dim

    def build(self, input_shape):

        self.pos_embed = self.add_weight(
            shape=(1, self.num_patches + 1, self.embed_dim),
            initializer="random_normal",
            trainable=True
        )

        self.cls_token = self.add_weight(
            shape=(1,1,self.embed_dim),
            initializer="random_normal",
            trainable=True
        )

    def call(self, x):

        batch = tf.shape(x)[0]

        cls = tf.repeat(self.cls_token, repeats=batch, axis=0)

        x = tf.concat([cls, x], axis=1)

        return x + self.pos_embed


class TransformerEncoder(layers.Layer):

    def __init__(self, embed_dim, num_heads, mlp_dim):
        super().__init__()

        self.norm1 = layers.LayerNormalization()

        self.attn = layers.MultiHeadAttention(
            num_heads=num_heads,
            key_dim=embed_dim // num_heads
        )

        self.dropout1 = layers.Dropout(0.2)

        self.norm2 = layers.LayerNormalization()

        self.mlp = tf.keras.Sequential([
            layers.Dense(mlp_dim, activation="gelu"),
            layers.Dropout(0.2),
            layers.Dense(embed_dim)
        ])

    def call(self, x):

        x = x + self.dropout1(
            self.attn(self.norm1(x), self.norm1(x))
        )

        x = x + self.mlp(self.norm2(x))

        return x


def build_vit():

    inputs = layers.Input((224,224,1))

    x = PatchEmbedding(PATCH_SIZE, EMBED_DIM)(inputs)

    x = AddPositionEmbedding(NUM_PATCHES, EMBED_DIM)(x)

    for _ in range(4):
        x = TransformerEncoder(EMBED_DIM, 8, 512)(x)

    x = layers.LayerNormalization()(x)

    x = x[:,0]

    x = layers.Dense(128, activation="relu")(x)

    x = layers.Dropout(0.4)(x)

    outputs = layers.Dense(2, activation="softmax")(x)

    model = tf.keras.Model(inputs, outputs)

    return model


model = build_vit()

model.summary()

Model: "functional_6"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ input_layer_2 (InputLayer)      │ (None, 224, 224, 1)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ patch_embedding                 │ (None, None, 256)      │        65,792 │
│ (PatchEmbedding)                │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ add_position_embedding          │ (None, 197, 256)       │        50,688 │
│ (AddPositionEmbedding)          │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ transformer_encoder             │ (None, 197, 256)       │       527,104 │
│ (TransformerEncoder)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ transformer_encoder_1           │ (None, 197, 256)       │       527,104 │
│ (TransformerEncoder)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ transformer_encoder_2           │ (None, 197, 256)       │       527,104 │
│ (TransformerEncoder)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ transformer_encoder_3           │ (None, 197, 256)       │       527,104 │
│ (TransformerEncoder)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ layer_normalization_8           │ (None, 197, 256)       │           512 │
│ (LayerNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ get_item (GetItem)              │ (None, 256)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_15 (Dense)                │ (None, 128)            │        32,896 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_12 (Dropout)            │ (None, 128)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_16 (Dense)                │ (None, 2)              │           258 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 2,258,562 (8.62 MB)

 Trainable params: 2,258,562 (8.62 MB)

 Non-trainable params: 0 (0.00 B)

In [27]:
model.compile(
    optimizer=tf.keras.optimizers.AdamW(
        learning_rate=1e-5,
        weight_decay=1e-4
    ),
    loss="sparse_categorical_crossentropy",
    metrics=["accuracy"]
)

In [28]:
history = model.fit(

    train_ds,

    validation_data = val_ds,

    epochs = 30,

    class_weight = class_weights

)

Epoch 1/30
37/37 ━━━━━━━━━━━━━━━━━━━━ 51s 651ms/step - accuracy: 0.6436 - loss: 0.6506 - val_accuracy: 0.8082 - val_loss: 0.6840
Epoch 2/30
37/37 ━━━━━━━━━━━━━━━━━━━━ 7s 184ms/step - accuracy: 0.5852 - loss: 0.6994 - val_accuracy: 0.8082 - val_loss: 0.6855
Epoch 3/30
37/37 ━━━━━━━━━━━━━━━━━━━━ 10s 177ms/step - accuracy: 0.6697 - loss: 0.6708 - val_accuracy: 0.8082 - val_loss: 0.6851
Epoch 4/30
37/37 ━━━━━━━━━━━━━━━━━━━━ 8s 225ms/step - accuracy: 0.5737 - loss: 0.7324 - val_accuracy: 0.8082 - val_loss: 0.6922
Epoch 5/30
37/37 ━━━━━━━━━━━━━━━━━━━━ 7s 176ms/step - accuracy: 0.5400 - loss: 0.6511 - val_accuracy: 0.8082 - val_loss: 0.6903
Epoch 6/30
37/37 ━━━━━━━━━━━━━━━━━━━━ 8s 222ms/step - accuracy: 0.5649 - loss: 0.6785 - val_accuracy: 0.8082 - val_loss: 0.6893
Epoch 7/30
37/37 ━━━━━━━━━━━━━━━━━━━━ 7s 175ms/step - accuracy: 0.5295 - loss: 0.7082 - val_accuracy: 0.8082 - val_loss: 0.6927
Epoch 8/30
37/37 ━━━━━━━━━━━━━━━━━━━━ 8s 214ms/step - accuracy: 0.5327 - loss: 0.6785 - val_accuracy: 